# Black-Scholes-Merton Pricing and Greeks

This notebook validates European option pricing, put-call parity and Greek conventions using the reusable package under `src/`.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
import plotly.graph_objects as go

from derivatives_engine.models.black_scholes import call_price, put_price, put_call_parity_error
from derivatives_engine.risk.greeks import greek_table, finite_difference_delta, finite_difference_gamma, finite_difference_vega, finite_difference_theta, finite_difference_rho

In [ ]:
S, K, T, r, q, sigma = 100.0, 100.0, 1.0, 0.05, 0.0, 0.20
call = call_price(S, K, T, r, q, sigma)
put = put_price(S, K, T, r, q, sigma)
pd.DataFrame([
    {'instrument': 'European call', 'price': call},
    {'instrument': 'European put', 'price': put},
    {'instrument': 'put-call parity error', 'price': put_call_parity_error(S, K, T, r, q, sigma)},
])

In [ ]:
greeks = greek_table(S, K, T, r, q, sigma, 'call')
numerical = {
    'delta': finite_difference_delta(S, K, T, r, q, sigma, 'call'),
    'gamma': finite_difference_gamma(S, K, T, r, q, sigma, 'call'),
    'vega': finite_difference_vega(S, K, T, r, q, sigma, 'call'),
    'theta_annual': finite_difference_theta(S, K, T, r, q, sigma, 'call'),
    'rho': finite_difference_rho(S, K, T, r, q, sigma, 'call'),
}
pd.DataFrame([
    {'greek': key, 'analytical': greeks[key], 'finite_difference': numerical[key], 'absolute_error': abs(greeks[key] - numerical[key])}
    for key in numerical
])

In [ ]:
spots = np.linspace(60, 140, 100)
rows = []
for spot in spots:
    row = greek_table(float(spot), K, T, r, q, sigma, 'call')
    rows.append({'spot': spot, 'delta': row['delta'], 'gamma': row['gamma'], 'vega': row['vega']})
greek_df = pd.DataFrame(rows)
fig = go.Figure()
for column in ['delta', 'gamma', 'vega']:
    fig.add_trace(go.Scatter(x=greek_df['spot'], y=greek_df[column], name=column))
fig.update_layout(title='Call Greeks vs Spot', xaxis_title='Spot', yaxis_title='Greek value', template='plotly_white')
fig.show()